# End-to-End Lakehouse Pipeline (PoC)

This notebook demonstrates the Medallion Architecture flow based on our `senior-data-engineer` and `spark-optimization` principles:
1. **Bronze (Raw)**: Extract data via ESPN API and save raw JSON directly to MinIO, ensuring an immutable history.
2. **Silver (Analytics)**: Use PySpark to read the Bronze JSON, apply optimial transformations (Goal Diff, Total Goals), and save as an idempotent Apache Iceberg table.

### 0. Install Dependencies

In [ ]:
# Install Boto3 for MinIO interaction and soccerdata for the ESPN extraction
!pip install -q boto3 soccerdata pandas

### 1. Extract Data (Simulating Airflow Task -> Bronze)

In [ ]:
import os
import json
import time
import boto3
import soccerdata as sd
from soccerdata import _config

LEAGUE = "BRA-Brasileirao"
SEASON = 2024

# Ensure the league mapping exists
if LEAGUE not in _config.LEAGUE_DICT:
    _config.LEAGUE_DICT[LEAGUE] = {"ESPN": "bra.1"}

# Initialize ESPN reader
print(f"Fetching schedule for {LEAGUE} {SEASON}...")
reader = sd.ESPN(leagues=LEAGUE, seasons=SEASON)
schedule = reader.read_schedule().reset_index()
print(f"Extracted {len(schedule)} matches.")

# Convert to JSON
schedule_json = schedule.to_json(orient="records", date_format="iso")
schedule_records = json.loads(schedule_json)
schedule_json_str = json.dumps(schedule_records, indent=2, ensure_ascii=False)


### 2. Upload to MinIO (Bronze Layer)

In [ ]:
# Connect to MinIO internal network via Boto3
MINIO_ENDPOINT = "http://minio:9000" 
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
RAW_BUCKET = "datalake-raw"

s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1"
)

bronze_path = f"espn/brasileirao/{SEASON}/matches.json"

# Upload to MinIO (Simulating the end of an Airflow Extract Task)
s3_client.put_object(
    Bucket=RAW_BUCKET,
    Key=bronze_path,
    Body=schedule_json_str
)
print(f"Successfully uploaded raw data to s3://{RAW_BUCKET}/{bronze_path}")

### 3. Spark Initialization & Optimization

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# The Spark session is already configured via spark-defaults.conf to use Iceberg and connect to MinIO.
# Applying Adaptive Query Execution (AQE) best practices for optimizations.
spark = (SparkSession.builder
    .appName("BrasileiraoLakehouseAnalytics")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created with AQE enabled.")

### 4. PySpark Normalization (Bronze -> Silver)
Read the JSON directly from the Lake.

In [ ]:
# Read the Bronze JSON from MinIO using S3A
bronze_uri = f"s3a://{RAW_BUCKET}/{bronze_path}"
df_bronze = spark.read.option("multiline", "true").json(bronze_uri)

print(f"Read {df_bronze.count()} records from Bronze.")

def ensure_column(df, col_name, default_val, col_type):
    if col_name not in df.columns:
        return df.withColumn(col_name, F.lit(default_val).cast(col_type))
    return df

df_silver = df_bronze

# ESPN often puts the score inside the "home_score" / "away_score" fields, 
# but sometimes it extracts it as "home_team_score" / "away_team_score" depending on the API payload.
# Let's use coalesce to try to find the score in any of those forms:
if "home_team_score" in df_silver.columns and "home_score" not in df_silver.columns:
    df_silver = df_silver.withColumn("home_score", F.col("home_team_score"))
if "away_team_score" in df_silver.columns and "away_score" not in df_silver.columns:
    df_silver = df_silver.withColumn("away_score", F.col("away_team_score"))


# Finally ensure they exist even if everything is completely missing
df_silver = ensure_column(df_silver, "home_score", None, "int")
df_silver = ensure_column(df_silver, "away_score", None, "int")

# Apply Business Logic & Create Analytics Metrics
df_silver = (df_silver
    .withColumn("home_score", F.col("home_score").cast("int"))
    .withColumn("away_score", F.col("away_score").cast("int"))
    .withColumn("total_goals", F.col("home_score") + F.col("away_score"))
    .withColumn("goal_diff", F.abs(F.col("home_score") - F.col("away_score")))
    .withColumn("is_draw", F.col("home_score") == F.col("away_score"))
    .withColumn("processed_at", F.current_timestamp())
)

df_silver = df_silver.select(
        "game_id", "date", "game", "home_team", "home_score", 
        "away_team", "away_score", "total_goals", "goal_diff", "is_draw", "processed_at"
)

df_silver.show(5, truncate=False)

### 5. Write to Iceberg (Silver/Warehouse Layer)
Enforce idempotency with the `overwrite` mode so re-running this DAG doesn't duplicate data.

In [ ]:
# Create namespace if not exists
spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.analytics")

table_name = "lake.analytics.match_statistics"

# Write Data to Iceberg using Idempotent Overwrite
(df_silver.write
    .format("iceberg")
    .mode("overwrite")
    .save(table_name)
)

print(f"Successfully wrote to Iceberg table {table_name}")

### 6. Verify via SQL

In [ ]:
# Query the Iceberg table to see the results and schema
spark.sql(f"""
    SELECT game, total_goals, goal_diff, is_draw 
    FROM {table_name} 
    ORDER BY date DESC 
    LIMIT 10
""").show(truncate=False)